### Das Damenproblem (siehe [https://de.wikipedia.org/wiki/Damenproblem](https://de.wikipedia.org/wiki/Damenproblem))

Ziel ist es, alle Möglichkeiten zu finde, $n$ Damen so auf einem $nxn$-Schachbrett so zu platzieren, dass keine Dame eine andere schlagen kann. Das kleinste Brett auf dem dieses Problem lösbar ist, ist ein 
4x4 Brett. Eine Lösung ist unten gezeigt.

<img src='/files/images/4queens.png'>  

Der Suchraum besteht hier aus allen Teillösungen. Eine Teillösung ist ein Tuple der Form  
`queens = (1, 3, 0)`, also Damen auf den Feldern a2, b4 und c1, die sich alle nicht attackieren/sehen, bez. Damen auf den Feldern `[(col, row) for col, row in enumerate(queens)]`.

Die Nachbarn einer Teillösung `queens` sind die Teillösungen der Form `queens + (row,)`.  
Die Suche beginnt mit der leeren Teillösung `()` und endet bei einer
Teillösung `queens` mit `len(queens) == n`.  

Hier ist die Suche besonders einfach: Die Teillösungen bilden einen Baum. Die
Nachbarn eines noch unbesuchten Knotens sind immer verschieden
von allen bereits besuchten Knoten.
Kein Dict `go_back` oder Menge `visited` wird benötigt.


Eine Dame auf Feld (x1, y1) sieht eine Dame auf Feld (x2, y2), falls 
- x1 == x2 (gleiche Spalte), oder
- y1 == y2 (gleiche Reihe) oder
- abs(x2-x1) == abs(y2-y1) (gleiche Diagonale)

<div>
    <img src='/files/images/queens_1.svg'>
    <img src='/files/images/queens_2.svg'>
</div> 

### Aufgabe
1. Finde eine erste Lösung des 8-Damenproblems. Benutze einmal depth-first und einmal breadt-first Search.
   Wieviele Knoten werden dabei besucht, bez. in die Liste `nodes_to_visit` aufgenommen?
2. Finde alle Lösungen des 8-Damenproblems. Benutze einmal depth-first und einmal breadth-first Search.

**Bemerkungen**:
- Es gibt insgesamt 2057 Teillösungen und 92 Lösungen (Teillösungen der Länge 8).
- Breadth-First Search besucht zuerst die Teillösungen der Längen 1, 2, ..., 7.
  Beim Besuch der Teillösungen der Länge 7 werden dann die Teillösungen der Länge 8 von links an  `nodes_to_visit` angehängt.
  Dann werden diese besucht (Knoten 1966 bis 2057).
  Ab der 1. Lösungen sind alle weiteren besuchten Knoten Lösungen.

In [ ]:
from collections import deque


def search_df(node, get_successors, n_queens):
    nodes_to_visit = [node]
    go_back = {node: None}
    n_scheduled = 1
    n_visited = 0
    while nodes_to_visit:
        n_visited += 1
        node = nodes_to_visit.pop()
        if len(node) == n_queens:
            print(f'nodes scheduled for visiting: {n_scheduled}, nodes visited: {n_visited}')
            return go_back, node

        for neighbor in get_successors(node):
            n_scheduled += 1
            go_back[neighbor] = node
            nodes_to_visit.append(neighbor)

    print('Keine Loesung gefunden')
    print(f'nodes scheduled for visiting: {n_scheduled}, nodes visited: {n_visited}')
    return go_back, None


def search_bf(node, get_successors, n_queens):
    nodes_to_visit = deque([node])
    n_scheduled = 1
    n_visited = 0
    while nodes_to_visit:
        n_visited += 1
        node = nodes_to_visit.pop()
        if len(node) == n_queens:
            print(f'nodes scheduled for visiting: {n_scheduled}, nodes visited: {n_visited}')
            return node

        for neighbor in get_successors(node):
            n_scheduled += 1
            nodes_to_visit.appendleft(neighbor)

    print('Keine Loesung gefunden')
    print(f'nodes scheduled for visiting: {n_scheduled}, nodes visited: {n_visited}')


def search_df_all(node, get_successors, n_queens):
    nodes_to_visit = [node]
    n_scheduled = 1
    n_visited = 0
    while nodes_to_visit:
        n_visited += 1
        node = nodes_to_visit.pop()
        if len(node) == n_queens:
            print(f'nodes scheduled for visiting: {n_scheduled}, nodes visited: {n_visited}')
            yield node

        for neighbor in get_successors(node):
            n_scheduled += 1
            nodes_to_visit.append(neighbor)

    print(f'nodes scheduled for visiting: {n_scheduled}, nodes visited: {n_visited}')


def search_bf_all(node, get_successors, n_queens):
    nodes_to_visit = deque([node])
    n_scheduled = 1
    n_visited = 0
    while nodes_to_visit:
        n_visited += 1
        node = nodes_to_visit.pop()
        if len(node) == n_queens:
            print(f'nodes scheduled for visiting: {n_scheduled}, nodes visited: {n_visited}')
            yield node

        for neighbor in get_successors(node):
            n_scheduled += 1
            nodes_to_visit.appendleft(neighbor)

    print(f'nodes scheduled for visiting: {n_scheduled}, nodes visited: {n_visited}')

In [ ]:
def is_queen_move(p, q):
    '''p, q: Schachbrettfelder (col row)
       returns True falls eine Dame von p nach q ziehen kann
    '''
    return (p[0] == q[0]
            or p[1] == q[1]
            or abs(p[0] - q[0]) == abs(p[1] - q[1])
            )


def extend_queens(queens, n_queens=8):
    '''queens: Teilloesung der Form (0, 3, 5)
       liefert alle Teilloesungen der Form queens + (row,)
    '''
    col = len(queens)
    if col == n_queens:
        return

    for row in range(n_queens):
        if any(is_queen_move(p, (col, row)) for p in enumerate(queens)):
            continue

        yield queens + (row,)

In [ ]:
go_back, solution = search_df((), extend_queens, 8)
solution, len(go_back)

In [ ]:
# beachte: 1966 besuchte Knoten (von 2057) 
# die 92 Knoten 1966, ..., 2057 sind alles Loesungen
solution = search_bf((), extend_queens, 8)
solution

In [ ]:
solutions = list(search_df_all((), extend_queens, 8))

In [ ]:
solutions = list(search_bf_all((), extend_queens, 8))

In [ ]:
# besuche alle Knoten (es gibt keine Loesung der Laenge 9!)
go_back, solution = search_df((), extend_queens, 9)
len(go_back)

In [ ]:
# Wieviele Teilloesungen der Laenge n gibt es?
# erstelle einen Dict {n: Liste mit Teilloesungen der Laenge n, ...}

d = {}
for v in go_back:
    n = len(v)
    if n not in d:
        d[n] = [v]
    else:
        d[n].append(v)

In [ ]:
tot = 0
for k, vs in d.items():
    print(f'Anzahl Teilloesugen der Laenge {k}: {len(vs)}')
    tot += len(vs)
print(f'totale Anzahl Teillosungen: {tot}')